# < Model 1. baseline >
category, 
amt, 
trans_hour, 
age

: 별도의 개인화.행동 파생변수 없이 현재 거래의 기본 정보만 사용한 모델>

- 먼저 Baseline 70:30 모델에서 공통 고정 임계값을 한 번 결정

In [2]:
%pip install lightgbm

  Using cached narwhals-2.24.0-py3-none-any.whl.metadata (15 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   -- ------------------------------------- 0.1/1.4 MB 919.0 kB/s eta 0:00:02
   ---------- ----------------------------- 0.4/1.4 MB 2.9 MB/s eta 0:00:01
   ------------------------------------- -- 1.3/1.4 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------  1.4/1.4 MB 7.1 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 5.1 MB/s eta 0:00:00
Using cached narwhals-2.24.0-py3-none-any.whl (461 kB)
Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl (36.6 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import lightgbm as lgb

print("LightGBM 버전:", lgb.__version__)

LightGBM 버전: 4.7.0


In [2]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp311-cp311-win_amd64.whl (8.3 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import sklearn
print("scikit-learn 버전:", sklearn.__version__)

scikit-learn 버전: 1.9.0


In [1]:
import lightgbm
import sklearn

print(lightgbm.__version__)
print(sklearn.__version__)

4.7.0
1.9.0


In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    precision_recall_curve
)


# =========================================================
# 1. 데이터 불러오기
# =========================================================
csv_path = (
    r"C:\Users\splen\OneDrive\Desktop\BDAI_"
    r"\BOOSTMAP\Fraud-FDS-Project\data"
    r"\fraud_full_features.csv"
)

df = pd.read_csv(
    csv_path,
    parse_dates=["trans_date_trans_time"]
)

df = (
    df.sort_values("trans_date_trans_time")
      .reset_index(drop=True)
)

print("전체 데이터:", df.shape)


# =========================================================
# 2. Baseline 설명변수
# =========================================================
baseline_features = [
    "category",
    "amt",
    "trans_hour",
    "age"
]

target = "is_fraud"

X = df[baseline_features].copy()
y = df[target].astype("int8").copy()

# LightGBM 범주형 변수 지정
X["category"] = X["category"].astype("category")


# =========================================================
# 3. 시간순 70:30 분할
# =========================================================
split_index = int(len(df) * 0.70)

X_train = X.iloc[:split_index].copy()
y_train = y.iloc[:split_index].copy()

X_valid = X.iloc[split_index:].copy()
y_valid = y.iloc[split_index:].copy()

print("\nTrain:", X_train.shape)
print("Valid:", X_valid.shape)

print(
    "Train 기간:",
    df.loc[:split_index - 1, "trans_date_trans_time"].min(),
    "~",
    df.loc[:split_index - 1, "trans_date_trans_time"].max()
)

print(
    "Valid 기간:",
    df.loc[split_index:, "trans_date_trans_time"].min(),
    "~",
    df.loc[split_index:, "trans_date_trans_time"].max()
)

print("\nTrain 이상거래 건수:", int(y_train.sum()))
print("Valid 이상거래 건수:", int(y_valid.sum()))

print("Train 이상거래율:", y_train.mean())
print("Valid 이상거래율:", y_valid.mean())


# =========================================================
# 4. 클래스 불균형 가중치
# =========================================================
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())

scale_pos_weight = (
    negative_count / positive_count
)

print("\n정상거래 수:", negative_count)
print("이상거래 수:", positive_count)
print("scale_pos_weight:", scale_pos_weight)


# =========================================================
# 5. 모델 비교용 공통 하이퍼파라미터
# =========================================================
MODEL_PARAMS = {
    "objective": "binary",
    "boosting_type": "gbdt",

    "n_estimators": 1000,
    "learning_rate": 0.05,

    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 50,

    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,

    "reg_alpha": 0.0,
    "reg_lambda": 0.0,

    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

baseline_model_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight
)


# =========================================================
# 6. 학습
# LightGBM 4.7.0 방식으로 eval_X, eval_y 사용
# =========================================================
baseline_model_7030.fit(
    X_train,
    y_train,

    eval_X=X_valid,
    eval_y=y_valid,

    eval_metric="average_precision",
    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# =========================================================
# 7. 검증셋 예측확률
# =========================================================
valid_prob = baseline_model_7030.predict_proba(
    X_valid,
    num_iteration=baseline_model_7030.best_iteration_
)[:, 1]


# =========================================================
# 8. Baseline 70:30에서 공통 임계값 1회 선정
# =========================================================
precisions, recalls, thresholds = precision_recall_curve(
    y_valid,
    valid_prob
)

f1_scores = (
    2 * precisions[:-1] * recalls[:-1]
    / (
        precisions[:-1]
        + recalls[:-1]
        + 1e-12
    )
)

best_index = int(np.argmax(f1_scores))

FIXED_THRESHOLD = float(
    thresholds[best_index]
)

valid_pred = (
    valid_prob >= FIXED_THRESHOLD
).astype("int8")


# =========================================================
# 9. 성능 계산
# =========================================================
result_7030 = {
    "model": "Baseline",
    "split": "70:30",
    "best_iteration": baseline_model_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid,
        valid_prob
    ),

    "roc_auc": roc_auc_score(
        y_valid,
        valid_prob
    ),

    "precision": precision_score(
        y_valid,
        valid_pred,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid,
        valid_pred,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid,
        valid_pred,
        zero_division=0
    )
}


# =========================================================
# 10. 결과 출력
# =========================================================
print("\n========== Baseline 70:30 재실험 ==========")

print(
    "Best iteration :",
    result_7030["best_iteration"]
)

print(
    "고정 임계값     :",
    round(result_7030["threshold"], 6)
)

print(
    "PR-AUC         :",
    round(result_7030["pr_auc"], 6)
)

print(
    "ROC-AUC        :",
    round(result_7030["roc_auc"], 6)
)

print(
    "Precision      :",
    round(result_7030["precision"], 6)
)

print(
    "Recall         :",
    round(result_7030["recall"], 6)
)

print(
    "F1-score       :",
    round(result_7030["f1"], 6)
)

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid,
        valid_pred
    )
)

print("\n공통 하이퍼파라미터")
for key, value in MODEL_PARAMS.items():
    print(f"{key}: {value}")

print(
    "scale_pos_weight:",
    scale_pos_weight
)

전체 데이터: (1296675, 31)

Train: (907672, 4)
Valid: (389003, 4)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.867393	valid_0's binary_logloss: 0.0598627
[100]	valid_0's average_precision: 0.88679	valid_0's binary_logloss: 0.0490198
[150]	valid_0's average_precision: 0.890785	valid_0's binary_logloss: 0.0433696
[200]	valid_0's average_precision: 0.895938	valid_0's binary_logloss: 0.0394014
[250]	valid_0's average_precision: 0.898471	valid_0's binary_logloss: 0.0365787
[300]	valid_0's average_precision: 0.899856	valid_0's binary_logloss: 0.0342252
[350]	valid_0's average_precision: 0.900194	valid_0's binary_logloss: 0.0325393
[400]	valid_0's average_precision: 0.90

(907672, 4)에서 4는 설명변수 개수.

[50] <- 50번째 트리까지 학습.

early_stopping으로 450개 트리까지 학습, best iteration은 403번째 트리.

Confusion Matrix를 보면,

실제 이상거래 2,385(483 + 1902)건 중, 1902건 탐지, 483건 미탐지

정상거래 중 226건을 이상거래로 오탐

- 80:20 비교 (임계값 0.9900239로 쓰기, scale_pos_weight만 80% 학습셋 기준으로 재계산)

In [5]:
# =========================================================
# Baseline 80:20 비교
# =========================================================

FIXED_THRESHOLD = 0.990239

# 1. 시간순 80:20 분할
split_index_8020 = int(len(df) * 0.80)

X_train_8020 = X.iloc[:split_index_8020].copy()
y_train_8020 = y.iloc[:split_index_8020].copy()

X_valid_8020 = X.iloc[split_index_8020:].copy()
y_valid_8020 = y.iloc[split_index_8020:].copy()

print("Train:", X_train_8020.shape)
print("Valid:", X_valid_8020.shape)

print(
    "Train 기간:",
    df.loc[:split_index_8020 - 1, "trans_date_trans_time"].min(),
    "~",
    df.loc[:split_index_8020 - 1, "trans_date_trans_time"].max()
)

print(
    "Valid 기간:",
    df.loc[split_index_8020:, "trans_date_trans_time"].min(),
    "~",
    df.loc[split_index_8020:, "trans_date_trans_time"].max()
)

print("\nTrain 이상거래 건수:", int(y_train_8020.sum()))
print("Valid 이상거래 건수:", int(y_valid_8020.sum()))

print("Train 이상거래율:", y_train_8020.mean())
print("Valid 이상거래율:", y_valid_8020.mean())


# 2. 80% 학습셋 기준 scale_pos_weight 계산
negative_count_8020 = int((y_train_8020 == 0).sum())
positive_count_8020 = int((y_train_8020 == 1).sum())

scale_pos_weight_8020 = (
    negative_count_8020 / positive_count_8020
)

print("\n정상거래 수:", negative_count_8020)
print("이상거래 수:", positive_count_8020)
print("scale_pos_weight:", scale_pos_weight_8020)


# 3. 동일 하이퍼파라미터 모델 생성
baseline_model_8020 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_8020
)


# 4. 학습
baseline_model_8020.fit(
    X_train_8020,
    y_train_8020,

    eval_X=X_valid_8020,
    eval_y=y_valid_8020,

    eval_metric="average_precision",
    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 5. 예측확률
valid_prob_8020 = baseline_model_8020.predict_proba(
    X_valid_8020,
    num_iteration=baseline_model_8020.best_iteration_
)[:, 1]


# 6. 70:30에서 정한 고정 임계값 적용
valid_pred_8020 = (
    valid_prob_8020 >= FIXED_THRESHOLD
).astype("int8")


# 7. 평가
result_8020 = {
    "model": "Baseline",
    "split": "80:20",
    "best_iteration": baseline_model_8020.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_8020,
        valid_prob_8020
    ),

    "roc_auc": roc_auc_score(
        y_valid_8020,
        valid_prob_8020
    ),

    "precision": precision_score(
        y_valid_8020,
        valid_pred_8020,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_8020,
        valid_pred_8020,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_8020,
        valid_pred_8020,
        zero_division=0
    )
}


# 8. 결과 출력
print("\n========== Baseline 80:20 ==========")

print(
    "Best iteration :",
    result_8020["best_iteration"]
)

print(
    "고정 임계값     :",
    round(result_8020["threshold"], 6)
)

print(
    "PR-AUC         :",
    round(result_8020["pr_auc"], 6)
)

print(
    "ROC-AUC        :",
    round(result_8020["roc_auc"], 6)
)

print(
    "Precision      :",
    round(result_8020["precision"], 6)
)

print(
    "Recall         :",
    round(result_8020["recall"], 6)
)

print(
    "F1-score       :",
    round(result_8020["f1"], 6)
)

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_8020,
        valid_pred_8020
    )
)

Train: (1037340, 4)
Valid: (259335, 4)
Train 기간: 2019-01-01 00:00:18 ~ 2020-03-06 07:15:17
Valid 기간: 2020-03-06 07:16:43 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5968
Valid 이상거래 건수: 1538
Train 이상거래율: 0.005753176393467908
Valid 이상거래율: 0.005930553145545337

정상거래 수: 1031372
이상거래 수: 5968
scale_pos_weight: 172.8170241286863
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.874486	valid_0's binary_logloss: 0.0587517
[100]	valid_0's average_precision: 0.887648	valid_0's binary_logloss: 0.0486915
[150]	valid_0's average_precision: 0.892842	valid_0's binary_logloss: 0.0432684
[200]	valid_0's average_precision: 0.8968	valid_0's binary_logloss: 0.0395949
[250]	valid_0's average_precision: 0.898557	valid_0's binary_logloss: 0.0370391
[300]	valid_0's average_precision: 0.89888	valid_0's binary_logloss: 0.0349521
[350]	valid_0's average_precision: 0.900097	valid_0's binary_logloss: 0.0332299
[400]	valid_0's average_precision: 0.899874	valid_0's binary_logl

결과를 비교하면, baseline 모델은 70:30 분할 비율이 성능이 더 좋음.

이제부터 모든 모델을 두 번씩 돌리지 않고 70:30 한 번씩만 돌리기.

상위 2~3개 조합이 정해졌을 때 expanding window로 여러 미래 구간에서 안정성 확인.

# Model 2: 이진변수만 사용

- is_high_amt      : 거래금액이 500 이상인지
- high_speed       : 이동속도가 100 이상인지
- is_online        : 온라인 업종 거래인지
- risk_time_22_04  : 밤 10시~새벽 4시 거래인지

이 모델은 원본 변수를 이진 변수로 압축했을 때도 성능이 유지되는지 확인하는 모델

In [6]:
# =========================================================
# Model 2: 규칙형 변수 4개
# 시간순 70:30
# =========================================================

MODEL_NAME = "Rule-based only"
FIXED_THRESHOLD = 0.990239

model2_features = [
    "is_online",
    "is_high_amt",
    "risk_time_22_04",
    "high_speed"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model2_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model2_features))
print("사용 변수:", model2_features)


# 2. 설명변수와 목표변수 생성
X_model2 = df[model2_features].copy()
y_model2 = df["is_fraud"].astype("int8").copy()

print("\n변수 자료형")
print(X_model2.dtypes)

print("\n결측치 수")
print(X_model2.isna().sum())


# 3. 시간순 70:30 분할
split_index_model2 = int(len(df) * 0.70)

X_train_model2 = X_model2.iloc[:split_index_model2].copy()
X_valid_model2 = X_model2.iloc[split_index_model2:].copy()

y_train_model2 = y_model2.iloc[:split_index_model2].copy()
y_valid_model2 = y_model2.iloc[split_index_model2:].copy()

print("\nTrain:", X_train_model2.shape)
print("Valid:", X_valid_model2.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model2 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model2 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model2:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model2:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model2.sum()))
print("Valid 이상거래 건수:", int(y_valid_model2.sum()))

print("Train 이상거래율:", y_train_model2.mean())
print("Valid 이상거래율:", y_valid_model2.mean())


# 4. 클래스 불균형 가중치
negative_count_model2 = int((y_train_model2 == 0).sum())
positive_count_model2 = int((y_train_model2 == 1).sum())

scale_pos_weight_model2 = (
    negative_count_model2
    / positive_count_model2
)

print("\n정상거래 수:", negative_count_model2)
print("이상거래 수:", positive_count_model2)
print("scale_pos_weight:", scale_pos_weight_model2)


# 5. Baseline과 동일한 하이퍼파라미터로 모델 생성
model2_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model2
)


# 6. 학습
model2_7030.fit(
    X_train_model2,
    y_train_model2,

    eval_X=X_valid_model2,
    eval_y=y_valid_model2,

    eval_metric="average_precision",

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model2 = model2_7030.predict_proba(
    X_valid_model2,
    num_iteration=model2_7030.best_iteration_
)[:, 1]


# 8. Baseline에서 정한 고정 임계값 적용
valid_pred_model2 = (
    valid_prob_model2 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model2 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model2_features),
    "best_iteration": model2_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model2,
        valid_prob_model2
    ),

    "roc_auc": roc_auc_score(
        y_valid_model2,
        valid_prob_model2
    ),

    "precision": precision_score(
        y_valid_model2,
        valid_pred_model2,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model2,
        valid_pred_model2,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model2,
        valid_pred_model2,
        zero_division=0
    )
}


# 10. 결과 출력
print("\n========== Model 2: Rule-based only 70:30 ==========")

print("변수 수         :", result_model2["feature_count"])
print("Best iteration :", result_model2["best_iteration"])
print("고정 임계값     :", round(result_model2["threshold"], 6))
print("PR-AUC         :", round(result_model2["pr_auc"], 6))
print("ROC-AUC        :", round(result_model2["roc_auc"], 6))
print("Precision      :", round(result_model2["precision"], 6))
print("Recall         :", round(result_model2["recall"], 6))
print("F1-score       :", round(result_model2["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model2,
        valid_pred_model2
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model2["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model2["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model2["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model2["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model2["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 4
사용 변수: ['is_online', 'is_high_amt', 'risk_time_22_04', 'high_speed']

변수 자료형
is_online          int64
is_high_amt        int64
risk_time_22_04    int64
high_speed         int64
dtype: object

결측치 수
is_online          0
is_high_amt        0
risk_time_22_04    0
high_speed         0
dtype: int64

Train: (907672, 4)
Valid: (389003, 4)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.328726	valid_0's binary_logloss: 0.320069
Early stopping, best iteration is:
[2]	valid_0's average_precision: 0.329021	valid_0's binary_logloss: 0.284655
Evaluated only: average_precision

========== Model 2: Rule-based only 70:30 ==========
변수 수         : 4
Best iteration : 2

1. PR-AUC = 0.33 정도로, Baseline의 0.90보다 훨씬 낮음.

=> 이진 변수만 써서 원래 정보가 너무 많이 사라진 것.

2. Precision·Recall·F1이 모두 0 : 고정 임계값 0.990239를 넘는 예측확률이 하나도 없었기 때문.

3. confusion matrix도 모든 거래를 정상으로 분류함.

4. Best iteration=2 : 같은 맥락. 변수 조합이 단순해서 초반 이후 검증 PR-AUC가 거의 개선되지 않은 것.

=> 이 실험의 의미:

규칙형 "이진 변수만"으로는 원본 변수의 세부 정보를 충분히 대체하기 어렵다

# Model 3: Baseline + 이진변수

총 8가지 설명변수

이 모델은 원본 정보와 이진 정보를 함께 줬을 때 실제 성능이 추가로 개선되는지 확인.

In [7]:
# =========================================================
# Model 3: Baseline + 규칙형 변수 8개
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Rule-based"
FIXED_THRESHOLD = 0.990239

model3_features = [
    # Baseline
    "category",
    "amt",
    "trans_hour",
    "age",

    # 규칙형 변수
    "is_online",
    "is_high_amt",
    "risk_time_22_04",
    "high_speed"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model3_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model3_features))
print("사용 변수:", model3_features)


# 2. 설명변수와 목표변수 생성
X_model3 = df[model3_features].copy()
y_model3 = df["is_fraud"].astype("int8").copy()

# 범주형 변수 지정
X_model3["category"] = (
    X_model3["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model3.dtypes)

print("\n결측치 수")
print(X_model3.isna().sum())


# 3. 시간순 70:30 분할
split_index_model3 = int(len(df) * 0.70)

X_train_model3 = (
    X_model3
    .iloc[:split_index_model3]
    .copy()
)

X_valid_model3 = (
    X_model3
    .iloc[split_index_model3:]
    .copy()
)

y_train_model3 = (
    y_model3
    .iloc[:split_index_model3]
    .copy()
)

y_valid_model3 = (
    y_model3
    .iloc[split_index_model3:]
    .copy()
)

print("\nTrain:", X_train_model3.shape)
print("Valid:", X_valid_model3.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model3 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model3 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model3:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model3:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model3.sum()))
print("Valid 이상거래 건수:", int(y_valid_model3.sum()))

print("Train 이상거래율:", y_train_model3.mean())
print("Valid 이상거래율:", y_valid_model3.mean())


# 4. 클래스 불균형 가중치 계산
negative_count_model3 = int(
    (y_train_model3 == 0).sum()
)

positive_count_model3 = int(
    (y_train_model3 == 1).sum()
)

scale_pos_weight_model3 = (
    negative_count_model3
    / positive_count_model3
)

print("\n정상거래 수:", negative_count_model3)
print("이상거래 수:", positive_count_model3)
print("scale_pos_weight:", scale_pos_weight_model3)


# 5. 동일 하이퍼파라미터로 모델 생성
model3_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model3
)


# 6. 학습
model3_7030.fit(
    X_train_model3,
    y_train_model3,

    eval_X=X_valid_model3,
    eval_y=y_valid_model3,

    eval_metric="average_precision",

    categorical_feature=[
        "category"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model3 = model3_7030.predict_proba(
    X_valid_model3,
    num_iteration=model3_7030.best_iteration_
)[:, 1]


# 8. Baseline에서 정한 고정 임계값 적용
valid_pred_model3 = (
    valid_prob_model3 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model3 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model3_features),
    "best_iteration": model3_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model3,
        valid_prob_model3
    ),

    "roc_auc": roc_auc_score(
        y_valid_model3,
        valid_prob_model3
    ),

    "precision": precision_score(
        y_valid_model3,
        valid_pred_model3,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model3,
        valid_pred_model3,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model3,
        valid_pred_model3,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 3: "
    "Baseline + Rule-based 70:30 =========="
)

print("변수 수         :", result_model3["feature_count"])
print("Best iteration :", result_model3["best_iteration"])
print("고정 임계값     :", round(result_model3["threshold"], 6))
print("PR-AUC         :", round(result_model3["pr_auc"], 6))
print("ROC-AUC        :", round(result_model3["roc_auc"], 6))
print("Precision      :", round(result_model3["precision"], 6))
print("Recall         :", round(result_model3["recall"], 6))
print("F1-score       :", round(result_model3["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model3,
        valid_pred_model3
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model3["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model3["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model3["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model3["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model3["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 8
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'is_online', 'is_high_amt', 'risk_time_22_04', 'high_speed']

변수 자료형
category           category
amt                 float64
trans_hour            int64
age                   int64
is_online             int64
is_high_amt           int64
risk_time_22_04       int64
high_speed            int64
dtype: object

결측치 수
category           0
amt                0
trans_hour         0
age                0
is_online          0
is_high_amt        0
risk_time_22_04    0
high_speed         0
dtype: int64

Train: (907672, 8)
Valid: (389003, 8)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.866688	valid_0's binary_loglo

Model 3은 

PR-AUC와 Precision은 소폭 개선됐지만 

Recall과 F1이 하락했으므로, 

Baseline보다 확실히 우수하다고 보기 어렵다.

* 다만 Model 3을 바로 버리지는 x.

 PR-AUC가 더 높기 때문에 나중에 임계값을 다시 조정하면 Baseline보다 좋은 균형점이 나올 가능성 있음.  그래서 후보로는 남겨둘 수 있음.


# Model 4. Baseline + 개인별 금액 패턴 (설명변수 5개)

- amt_to_prior_median_ratio: 현재 금액이 평소 금액의 몇 배인지

이렇게 구성한 이유: 

amt_to어쩌구는 연속형 값이고 나머지 금액 패턴 관련 변수들보다 정보가 가장 풍부함.

has_prior어쩌구를 넣을지 말지 봤는데, NaN이 그대로 남아있어서 lightGBM이 알아서 처리하므로, 별도의 flag 변수 추가하면 같은 정보를 중복 제공하게 되는 것. 따라서 깔끔하게 변수 하나만 추가하는 걸로.

In [8]:
## amt_to어쩌구에 결측치 있는지 확인.

# amt_to_prior_median_ratio 결측치 확인

col = "amt_to_prior_median_ratio"

print("전체 행 수:", len(df))
print("NaN 개수:", df[col].isna().sum())
print("NaN 비율:", df[col].isna().mean())

print("\nNaN 행 일부")
print(
    df.loc[
        df[col].isna(),
        [
            "trans_date_trans_time",
            "amt",
            "prior_normal_median_amt",
            "amt_to_prior_median_ratio",
            "has_prior_normal_transaction"
        ]
    ].head(10)
)

전체 행 수: 1296675
NaN 개수: 1649
NaN 비율: 0.0012717141920681745

NaN 행 일부
  trans_date_trans_time     amt  prior_normal_median_amt  \
0   2019-01-01 00:00:18    4.97                      NaN   
1   2019-01-01 00:00:44  107.23                      NaN   
2   2019-01-01 00:00:51  220.11                      NaN   
3   2019-01-01 00:01:16   45.00                      NaN   
4   2019-01-01 00:03:06   41.96                      NaN   
5   2019-01-01 00:04:08   94.63                      NaN   
6   2019-01-01 00:04:42   44.54                      NaN   
7   2019-01-01 00:05:08   71.65                      NaN   
8   2019-01-01 00:05:18    4.27                      NaN   
9   2019-01-01 00:06:01  198.39                      NaN   

   amt_to_prior_median_ratio  has_prior_normal_transaction  
0                        NaN                             0  
1                        NaN                             0  
2                        NaN                             0  
3                        N

In [9]:
print(
    "무한대 개수:",
    np.isinf(df["amt_to_prior_median_ratio"]).sum()
)

무한대 개수: 0


In [10]:
# =========================================================
# Model 4: Baseline + 개인별 금액 비율
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Personalized Amount Ratio"
FIXED_THRESHOLD = 0.990239

model4_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model4_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model4_features))
print("사용 변수:", model4_features)


# 2. 설명변수와 목표변수 생성
X_model4 = df[model4_features].copy()
y_model4 = df["is_fraud"].astype("int8").copy()

X_model4["category"] = (
    X_model4["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model4.dtypes)

print("\n결측치 수")
print(X_model4.isna().sum())

print(
    "\namt_to_prior_median_ratio 기초 통계"
)
print(
    X_model4[
        "amt_to_prior_median_ratio"
    ].describe()
)


# 3. 시간순 70:30 분할
split_index_model4 = int(len(df) * 0.70)

X_train_model4 = (
    X_model4
    .iloc[:split_index_model4]
    .copy()
)

X_valid_model4 = (
    X_model4
    .iloc[split_index_model4:]
    .copy()
)

y_train_model4 = (
    y_model4
    .iloc[:split_index_model4]
    .copy()
)

y_valid_model4 = (
    y_model4
    .iloc[split_index_model4:]
    .copy()
)

print("\nTrain:", X_train_model4.shape)
print("Valid:", X_valid_model4.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model4 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model4 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model4:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model4:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model4.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model4.sum())
)

print(
    "Train 이상거래율:",
    y_train_model4.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model4.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model4 = int(
    (y_train_model4 == 0).sum()
)

positive_count_model4 = int(
    (y_train_model4 == 1).sum()
)

scale_pos_weight_model4 = (
    negative_count_model4
    / positive_count_model4
)

print(
    "\n정상거래 수:",
    negative_count_model4
)

print(
    "이상거래 수:",
    positive_count_model4
)

print(
    "scale_pos_weight:",
    scale_pos_weight_model4
)


# 5. 동일 하이퍼파라미터로 모델 생성
model4_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model4
)


# 6. 학습
model4_7030.fit(
    X_train_model4,
    y_train_model4,

    eval_X=X_valid_model4,
    eval_y=y_valid_model4,

    eval_metric="average_precision",

    categorical_feature=[
        "category"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model4 = model4_7030.predict_proba(
    X_valid_model4,
    num_iteration=model4_7030.best_iteration_
)[:, 1]


# 8. Baseline에서 정한 고정 임계값 적용
valid_pred_model4 = (
    valid_prob_model4 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model4 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model4_features),
    "best_iteration": model4_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model4,
        valid_prob_model4
    ),

    "roc_auc": roc_auc_score(
        y_valid_model4,
        valid_prob_model4
    ),

    "precision": precision_score(
        y_valid_model4,
        valid_pred_model4,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model4,
        valid_pred_model4,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model4,
        valid_pred_model4,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 4: "
    "Baseline + Personalized Amount Ratio 70:30 =========="
)

print(
    "변수 수         :",
    result_model4["feature_count"]
)

print(
    "Best iteration :",
    result_model4["best_iteration"]
)

print(
    "고정 임계값     :",
    round(
        result_model4["threshold"],
        6
    )
)

print(
    "PR-AUC         :",
    round(
        result_model4["pr_auc"],
        6
    )
)

print(
    "ROC-AUC        :",
    round(
        result_model4["roc_auc"],
        6
    )
)

print(
    "Precision      :",
    round(
        result_model4["precision"],
        6
    )
)

print(
    "Recall         :",
    round(
        result_model4["recall"],
        6
    )
)

print(
    "F1-score       :",
    round(
        result_model4["f1"],
        6
    )
)

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model4,
        valid_pred_model4
    )
)


# 11. Baseline 대비 변화
print(
    "\n========== Baseline 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model4["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model4["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model4["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model4["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model4["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'amt_to_prior_median_ratio']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
amt_to_prior_median_ratio     float64
dtype: object

결측치 수
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
dtype: int64

amt_to_prior_median_ratio 기초 통계
count    1.295026e+06
mean     1.643860e+00
std      4.558937e+00
min      2.176430e-03
25%      2.606922e-01
50%      1.003543e+00
75%      1.894498e+00
max      8.557099e+02
Name: amt_to_prior_median_ratio, dtype: float64

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

Model4가 현재까지 가장 좋은 후보.

PR-AUC basline 대비 +0.016, precision +0.018, Recall -0.005

고정 임계값에서는 Recall이 약간 줄었지만 Precision 과 F1이 개선됨.

# Model 5. Baseline + 개인별 시간대 이탈

baseline + outside_trans_hours_80

In [11]:
# =========================================================
# Model 5: Baseline + 개인별 거래시간 이탈
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Outside Active Hours"
FIXED_THRESHOLD = 0.990239

model5_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "outside_trans_hours_80"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model5_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model5_features))
print("사용 변수:", model5_features)


# 2. 설명변수와 목표변수 생성
X_model5 = df[model5_features].copy()
y_model5 = df["is_fraud"].astype("int8").copy()

X_model5["category"] = (
    X_model5["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model5.dtypes)

print("\n결측치 수")
print(X_model5.isna().sum())

print("\noutside_trans_hours_80 값 분포")
print(
    X_model5["outside_trans_hours_80"]
    .value_counts(dropna=False)
    .sort_index()
)


# 3. 시간순 70:30 분할
split_index_model5 = int(len(df) * 0.70)

X_train_model5 = (
    X_model5
    .iloc[:split_index_model5]
    .copy()
)

X_valid_model5 = (
    X_model5
    .iloc[split_index_model5:]
    .copy()
)

y_train_model5 = (
    y_model5
    .iloc[:split_index_model5]
    .copy()
)

y_valid_model5 = (
    y_model5
    .iloc[split_index_model5:]
    .copy()
)

print("\nTrain:", X_train_model5.shape)
print("Valid:", X_valid_model5.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model5 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model5 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model5:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model5:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model5.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model5.sum())
)

print(
    "Train 이상거래율:",
    y_train_model5.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model5.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model5 = int(
    (y_train_model5 == 0).sum()
)

positive_count_model5 = int(
    (y_train_model5 == 1).sum()
)

scale_pos_weight_model5 = (
    negative_count_model5
    / positive_count_model5
)

print("\n정상거래 수:", negative_count_model5)
print("이상거래 수:", positive_count_model5)
print("scale_pos_weight:", scale_pos_weight_model5)


# 5. Baseline과 동일한 하이퍼파라미터로 모델 생성
model5_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model5
)


# 6. 학습
model5_7030.fit(
    X_train_model5,
    y_train_model5,

    eval_X=X_valid_model5,
    eval_y=y_valid_model5,

    eval_metric="average_precision",

    categorical_feature=[
        "category"
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model5 = model5_7030.predict_proba(
    X_valid_model5,
    num_iteration=model5_7030.best_iteration_
)[:, 1]


# 8. Baseline에서 정한 고정 임계값 적용
valid_pred_model5 = (
    valid_prob_model5 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model5 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model5_features),
    "best_iteration": model5_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model5,
        valid_prob_model5
    ),

    "roc_auc": roc_auc_score(
        y_valid_model5,
        valid_prob_model5
    ),

    "precision": precision_score(
        y_valid_model5,
        valid_pred_model5,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model5,
        valid_pred_model5,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model5,
        valid_pred_model5,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 5: "
    "Baseline + Outside Active Hours 70:30 =========="
)

print("변수 수         :", result_model5["feature_count"])
print("Best iteration :", result_model5["best_iteration"])
print("고정 임계값     :", round(result_model5["threshold"], 6))
print("PR-AUC         :", round(result_model5["pr_auc"], 6))
print("ROC-AUC        :", round(result_model5["roc_auc"], 6))
print("Precision      :", round(result_model5["precision"], 6))
print("Recall         :", round(result_model5["recall"], 6))
print("F1-score       :", round(result_model5["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model5,
        valid_pred_model5
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model5["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model5["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model5["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model5["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model5["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'outside_trans_hours_80']

변수 자료형
category                  category
amt                        float64
trans_hour                   int64
age                          int64
outside_trans_hours_80       int64
dtype: object

결측치 수
category                  0
amt                       0
trans_hour                0
age                       0
outside_trans_hours_80    0
dtype: int64

outside_trans_hours_80 값 분포
outside_trans_hours_80
0    1028541
1     268134
Name: count, dtype: int64

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.24506932239797
Training until validation scores don't improve for 50 rounds
[50]	valid_0's average_precision: 0.857803	valid_0's binary_logloss: 0.05

Model 5는 baseline 대비 성능 개선 거의 없음. 

단, outside_trans_hours_80은 정상거래를 더 보수적으로 거르는데는 조금 도움이 됨. 하지만 이상거래는 더 많이 놓침. 단독 추가 변수로서의 효과는 제한적.

# Model 6. Baseline + count_30min

In [12]:
# =========================================================
# Model 6: Baseline + 최근 30분 거래 횟수
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + 30min Transaction Count"
FIXED_THRESHOLD = 0.990239

model6_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "count_30min"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model6_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model6_features))
print("사용 변수:", model6_features)


# 2. 설명변수와 목표변수 생성
X_model6 = df[model6_features].copy()
y_model6 = df["is_fraud"].astype("int8").copy()

X_model6["category"] = X_model6["category"].astype("category")

print("\n변수 자료형")
print(X_model6.dtypes)

print("\n결측치 수")
print(X_model6.isna().sum())

print("\ncount_30min 기초 통계")
print(X_model6["count_30min"].describe())

print("\ncount_30min 값 분포 상위 15개")
print(
    X_model6["count_30min"]
    .value_counts(dropna=False)
    .sort_index()
    .head(15)
)


# 3. 시간순 70:30 분할
split_index_model6 = int(len(df) * 0.70)

X_train_model6 = X_model6.iloc[:split_index_model6].copy()
X_valid_model6 = X_model6.iloc[split_index_model6:].copy()

y_train_model6 = y_model6.iloc[:split_index_model6].copy()
y_valid_model6 = y_model6.iloc[split_index_model6:].copy()

print("\nTrain:", X_train_model6.shape)
print("Valid:", X_valid_model6.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model6 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model6 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model6:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model6:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model6.sum()))
print("Valid 이상거래 건수:", int(y_valid_model6.sum()))

print("Train 이상거래율:", y_train_model6.mean())
print("Valid 이상거래율:", y_valid_model6.mean())


# 4. 클래스 불균형 가중치 계산
negative_count_model6 = int((y_train_model6 == 0).sum())
positive_count_model6 = int((y_train_model6 == 1).sum())

scale_pos_weight_model6 = (
    negative_count_model6 / positive_count_model6
)

print("\n정상거래 수:", negative_count_model6)
print("이상거래 수:", positive_count_model6)
print("scale_pos_weight:", scale_pos_weight_model6)


# 5. 동일 하이퍼파라미터로 모델 생성
model6_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model6
)


# 6. 학습
model6_7030.fit(
    X_train_model6,
    y_train_model6,

    eval_X=X_valid_model6,
    eval_y=y_valid_model6,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model6 = model6_7030.predict_proba(
    X_valid_model6,
    num_iteration=model6_7030.best_iteration_
)[:, 1]


# 8. Baseline에서 정한 고정 임계값 적용
valid_pred_model6 = (
    valid_prob_model6 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model6 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model6_features),
    "best_iteration": model6_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model6,
        valid_prob_model6
    ),

    "roc_auc": roc_auc_score(
        y_valid_model6,
        valid_prob_model6
    ),

    "precision": precision_score(
        y_valid_model6,
        valid_pred_model6,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model6,
        valid_pred_model6,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model6,
        valid_pred_model6,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 6: "
    "Baseline + 30min Transaction Count 70:30 =========="
)

print("변수 수         :", result_model6["feature_count"])
print("Best iteration :", result_model6["best_iteration"])
print("고정 임계값     :", round(result_model6["threshold"], 6))
print("PR-AUC         :", round(result_model6["pr_auc"], 6))
print("ROC-AUC        :", round(result_model6["roc_auc"], 6))
print("Precision      :", round(result_model6["precision"], 6))
print("Recall         :", round(result_model6["recall"], 6))
print("F1-score       :", round(result_model6["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model6,
        valid_pred_model6
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model6["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model6["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model6["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model6["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model6["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'count_30min']

변수 자료형
category       category
amt             float64
trans_hour        int64
age               int64
count_30min       int64
dtype: object

결측치 수
category       0
amt            0
trans_hour     0
age            0
count_30min    0
dtype: int64

count_30min 기초 통계
count    1.296675e+06
mean     9.095595e-01
std      4.783908e-01
min      0.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      5.000000e+00
Name: count_30min, dtype: float64

count_30min 값 분포 상위 15개
count_30min
0     206282
1    1007030
2      78081
3       4939
4        321
5         22
Name: count, dtype: int64

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.2450

Model 6는 5보단 낫지만 4보다는 약함.

count_30min은 PR-AUC는 어느 정도 높였지만, 

고정 임계값에서는 이상거래를 더 놓쳐서(24건 증가) Recall 과 F1이 감소함.

# Model 7. baseline + rolling_sum_amt_1h

In [13]:
# =========================================================
# Model 7: Baseline + 최근 1시간 누적 거래금액
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + 1h Rolling Amount Sum"
FIXED_THRESHOLD = 0.990239

model7_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "rolling_sum_amt_1h"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model7_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model7_features))
print("사용 변수:", model7_features)


# 2. 설명변수와 목표변수 생성
X_model7 = df[model7_features].copy()
y_model7 = df["is_fraud"].astype("int8").copy()

X_model7["category"] = X_model7["category"].astype("category")

print("\n변수 자료형")
print(X_model7.dtypes)

print("\n결측치 수")
print(X_model7.isna().sum())

print("\nrolling_sum_amt_1h 기초 통계")
print(X_model7["rolling_sum_amt_1h"].describe())

print("\n무한대 개수")
print(
    np.isinf(
        X_model7["rolling_sum_amt_1h"]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model7 = int(len(df) * 0.70)

X_train_model7 = X_model7.iloc[:split_index_model7].copy()
X_valid_model7 = X_model7.iloc[split_index_model7:].copy()

y_train_model7 = y_model7.iloc[:split_index_model7].copy()
y_valid_model7 = y_model7.iloc[split_index_model7:].copy()

print("\nTrain:", X_train_model7.shape)
print("Valid:", X_valid_model7.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model7 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model7 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model7:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model7:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model7.sum()))
print("Valid 이상거래 건수:", int(y_valid_model7.sum()))

print("Train 이상거래율:", y_train_model7.mean())
print("Valid 이상거래율:", y_valid_model7.mean())


# 4. 클래스 불균형 가중치 계산
negative_count_model7 = int(
    (y_train_model7 == 0).sum()
)

positive_count_model7 = int(
    (y_train_model7 == 1).sum()
)

scale_pos_weight_model7 = (
    negative_count_model7
    / positive_count_model7
)

print("\n정상거래 수:", negative_count_model7)
print("이상거래 수:", positive_count_model7)
print("scale_pos_weight:", scale_pos_weight_model7)


# 5. 동일 하이퍼파라미터로 모델 생성
model7_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model7
)


# 6. 학습
model7_7030.fit(
    X_train_model7,
    y_train_model7,

    eval_X=X_valid_model7,
    eval_y=y_valid_model7,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model7 = model7_7030.predict_proba(
    X_valid_model7,
    num_iteration=model7_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model7 = (
    valid_prob_model7 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model7 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model7_features),
    "best_iteration": model7_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model7,
        valid_prob_model7
    ),

    "roc_auc": roc_auc_score(
        y_valid_model7,
        valid_prob_model7
    ),

    "precision": precision_score(
        y_valid_model7,
        valid_pred_model7,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model7,
        valid_pred_model7,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model7,
        valid_pred_model7,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 7: "
    "Baseline + 1h Rolling Amount Sum 70:30 =========="
)

print("변수 수         :", result_model7["feature_count"])
print("Best iteration :", result_model7["best_iteration"])
print("고정 임계값     :", round(result_model7["threshold"], 6))
print("PR-AUC         :", round(result_model7["pr_auc"], 6))
print("ROC-AUC        :", round(result_model7["roc_auc"], 6))
print("Precision      :", round(result_model7["precision"], 6))
print("Recall         :", round(result_model7["recall"], 6))
print("F1-score       :", round(result_model7["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model7,
        valid_pred_model7
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model7["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model7["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model7["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model7["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model7["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'rolling_sum_amt_1h']

변수 자료형
category              category
amt                    float64
trans_hour               int64
age                      int64
rolling_sum_amt_1h     float64
dtype: object

결측치 수
category              0
amt                   0
trans_hour            0
age                   0
rolling_sum_amt_1h    0
dtype: int64

rolling_sum_amt_1h 기초 통계
count    1.296675e+06
mean     8.491117e+01
std      1.947081e+02
min      1.000000e+00
25%      1.425000e+01
50%      5.429000e+01
75%      9.630000e+01
max      2.894890e+04
Name: rolling_sum_amt_1h, dtype: float64

무한대 개수
0

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weight: 176.24506932239797
Training until validation sco

Model 7은 현재까지 가장 좋은 모델.
Precision과 Recall 둘 다 개선됨.

오탐 44건 감소, 미탐 48건 감소, 탐지 성공 48건 증가.

# Model 8. baseline + recent_24h_high_amt_count

In [14]:
# =========================================================
# Model 8: Baseline + 최근 24시간 고액거래 횟수
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Recent 24h High Amount Count"
FIXED_THRESHOLD = 0.990239

model8_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model8_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model8_features))
print("사용 변수:", model8_features)


# 2. 설명변수와 목표변수 생성
X_model8 = df[model8_features].copy()
y_model8 = df["is_fraud"].astype("int8").copy()

X_model8["category"] = X_model8["category"].astype("category")

print("\n변수 자료형")
print(X_model8.dtypes)

print("\n결측치 수")
print(X_model8.isna().sum())

print("\nrecent_24h_high_amt_count 기초 통계")
print(
    X_model8["recent_24h_high_amt_count"].describe()
)

print("\nrecent_24h_high_amt_count 값 분포")
print(
    X_model8["recent_24h_high_amt_count"]
    .value_counts(dropna=False)
    .sort_index()
    .head(20)
)


# 3. 시간순 70:30 분할
split_index_model8 = int(len(df) * 0.70)

X_train_model8 = X_model8.iloc[:split_index_model8].copy()
X_valid_model8 = X_model8.iloc[split_index_model8:].copy()

y_train_model8 = y_model8.iloc[:split_index_model8].copy()
y_valid_model8 = y_model8.iloc[split_index_model8:].copy()

print("\nTrain:", X_train_model8.shape)
print("Valid:", X_valid_model8.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model8 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model8 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model8:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model8:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model8.sum()))
print("Valid 이상거래 건수:", int(y_valid_model8.sum()))

print("Train 이상거래율:", y_train_model8.mean())
print("Valid 이상거래율:", y_valid_model8.mean())


# 4. 클래스 불균형 가중치 계산
negative_count_model8 = int(
    (y_train_model8 == 0).sum()
)

positive_count_model8 = int(
    (y_train_model8 == 1).sum()
)

scale_pos_weight_model8 = (
    negative_count_model8
    / positive_count_model8
)

print("\n정상거래 수:", negative_count_model8)
print("이상거래 수:", positive_count_model8)
print("scale_pos_weight:", scale_pos_weight_model8)


# 5. 동일 하이퍼파라미터로 모델 생성
model8_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model8
)


# 6. 학습
model8_7030.fit(
    X_train_model8,
    y_train_model8,

    eval_X=X_valid_model8,
    eval_y=y_valid_model8,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model8 = model8_7030.predict_proba(
    X_valid_model8,
    num_iteration=model8_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model8 = (
    valid_prob_model8 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model8 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model8_features),
    "best_iteration": model8_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model8,
        valid_prob_model8
    ),

    "roc_auc": roc_auc_score(
        y_valid_model8,
        valid_prob_model8
    ),

    "precision": precision_score(
        y_valid_model8,
        valid_pred_model8,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model8,
        valid_pred_model8,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model8,
        valid_pred_model8,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 8: "
    "Baseline + Recent 24h High Amount Count 70:30 =========="
)

print("변수 수         :", result_model8["feature_count"])
print("Best iteration :", result_model8["best_iteration"])
print("고정 임계값     :", round(result_model8["threshold"], 6))
print("PR-AUC         :", round(result_model8["pr_auc"], 6))
print("ROC-AUC        :", round(result_model8["roc_auc"], 6))
print("Precision      :", round(result_model8["precision"], 6))
print("Recall         :", round(result_model8["recall"], 6))
print("F1-score       :", round(result_model8["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model8,
        valid_pred_model8
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model8["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model8["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model8["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model8["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model8["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
recent_24h_high_amt_count       int64
dtype: object

결측치 수
category                     0
amt                          0
trans_hour                   0
age                          0
recent_24h_high_amt_count    0
dtype: int64

recent_24h_high_amt_count 기초 통계
count    1.296675e+06
mean     4.816936e-02
std      2.741523e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      9.000000e+00
Name: recent_24h_high_amt_count, dtype: float64

recent_24h_high_amt_count 값 분포
recent_24h_high_amt_count
0    1244934
1      45913
2       3132
3       1342
4        792
5        348
6        161
7         42
8         10
9          1
Name: count, dtype: int64

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-

Model 8은 압도적으로 가장 좋음. 다만 성능 상승 폭이 매우 커서, 현재 거래나 미래 거래를 포함하지 않았는지 확인해야 함.

- 오탐 226 -> 98건
- 미탐 483 -> 267건
- 탐지 성공 : 1902 -> 2118건

In [15]:
# =========================================================
# recent_24h_high_amt_count 현재 거래 포함 여부 1차 점검
# =========================================================

check_df = (
    df.sort_values(
        ["cc_num", "trans_date_trans_time"]
    )
    .copy()
)

# 고객별 데이터상 최초 거래
first_transactions = (
    check_df
    .groupby("cc_num", observed=True)
    .head(1)
)

print("고객별 최초 거래 수:", len(first_transactions))

print("\n최초 거래의 recent_24h_high_amt_count 분포")
print(
    first_transactions[
        "recent_24h_high_amt_count"
    ]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n최초 거래인데 count가 1 이상인 행 수")
print(
    (
        first_transactions[
            "recent_24h_high_amt_count"
        ] >= 1
    ).sum()
)

print("\n최초 거래 일부")
print(
    first_transactions[
        [
            "cc_num",
            "trans_date_trans_time",
            "amt",
            "recent_24h_high_amt_count"
        ]
    ].head(20)
)

고객별 최초 거래 수: 983

최초 거래의 recent_24h_high_amt_count 분포
recent_24h_high_amt_count
0    983
Name: count, dtype: int64

최초 거래인데 count가 1 이상인 행 수
0

최초 거래 일부
               cc_num trans_date_trans_time     amt  recent_24h_high_amt_count
1017      60416207185   2019-01-01 12:47:15    7.27                          0
4415      60422928733   2019-01-03 18:38:26   94.20                          0
516       60423098130   2019-01-01 06:48:36    5.68                          0
586       60427851591   2019-01-01 07:36:27   78.80                          0
7892      60487002085   2019-01-06 03:23:55    9.37                          0
984       60490596305   2019-01-01 12:31:09   87.07                          0
53        60495593109   2019-01-01 00:39:43  122.86                          0
252      501802953619   2019-01-01 03:12:06    8.68                          0
1150401  501818133297   2020-04-24 22:09:30  977.70                          0
525      501828204849   2019-01-01 06:53:54    7.12      

다만 이 검사는 1차 확인. 

완전하게 보려면 임의의 고객 몇 명에 대해 실제 과거 24시간 고액거래 횟수와 변수값이 일치하는지 직접 재계산하는 검사가 가장 확실함. 

그래도 지금 단계에서는 Model 8을 유효 후보로 유지.

# Model 9. baseline + speed_2

In [16]:
# =========================================================
# Model 9: Baseline + 이전 거래 대비 이동속도
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Transaction Speed"
FIXED_THRESHOLD = 0.990239

model9_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "speed_2"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model9_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model9_features))
print("사용 변수:", model9_features)


# 2. 설명변수와 목표변수 생성
X_model9 = df[model9_features].copy()
y_model9 = df["is_fraud"].astype("int8").copy()

X_model9["category"] = X_model9["category"].astype("category")

print("\n변수 자료형")
print(X_model9.dtypes)

print("\n결측치 수")
print(X_model9.isna().sum())

print("\nspeed_2 기초 통계")
print(X_model9["speed_2"].describe())

print("\nspeed_2 무한대 개수")
print(
    np.isinf(
        X_model9["speed_2"]
    ).sum()
)

print("\nspeed_2 상위 분위수")
print(
    X_model9["speed_2"].quantile(
        [0.90, 0.95, 0.99, 0.999, 1.0]
    )
)


# 3. 시간순 70:30 분할
split_index_model9 = int(len(df) * 0.70)

X_train_model9 = X_model9.iloc[:split_index_model9].copy()
X_valid_model9 = X_model9.iloc[split_index_model9:].copy()

y_train_model9 = y_model9.iloc[:split_index_model9].copy()
y_valid_model9 = y_model9.iloc[split_index_model9:].copy()

print("\nTrain:", X_train_model9.shape)
print("Valid:", X_valid_model9.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model9 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model9 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model9:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model9:,
        "trans_date_trans_time"
    ].max()
)

print("\nTrain 이상거래 건수:", int(y_train_model9.sum()))
print("Valid 이상거래 건수:", int(y_valid_model9.sum()))

print("Train 이상거래율:", y_train_model9.mean())
print("Valid 이상거래율:", y_valid_model9.mean())


# 4. 클래스 불균형 가중치 계산
negative_count_model9 = int(
    (y_train_model9 == 0).sum()
)

positive_count_model9 = int(
    (y_train_model9 == 1).sum()
)

scale_pos_weight_model9 = (
    negative_count_model9
    / positive_count_model9
)

print("\n정상거래 수:", negative_count_model9)
print("이상거래 수:", positive_count_model9)
print("scale_pos_weight:", scale_pos_weight_model9)


# 5. 동일 하이퍼파라미터로 모델 생성
model9_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model9
)


# 6. 학습
model9_7030.fit(
    X_train_model9,
    y_train_model9,

    eval_X=X_valid_model9,
    eval_y=y_valid_model9,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model9 = model9_7030.predict_proba(
    X_valid_model9,
    num_iteration=model9_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model9 = (
    valid_prob_model9 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model9 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model9_features),
    "best_iteration": model9_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model9,
        valid_prob_model9
    ),

    "roc_auc": roc_auc_score(
        y_valid_model9,
        valid_prob_model9
    ),

    "precision": precision_score(
        y_valid_model9,
        valid_pred_model9,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model9,
        valid_pred_model9,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model9,
        valid_pred_model9,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 9: "
    "Baseline + Transaction Speed 70:30 =========="
)

print("변수 수         :", result_model9["feature_count"])
print("Best iteration :", result_model9["best_iteration"])
print("고정 임계값     :", round(result_model9["threshold"], 6))
print("PR-AUC         :", round(result_model9["pr_auc"], 6))
print("ROC-AUC        :", round(result_model9["roc_auc"], 6))
print("Precision      :", round(result_model9["precision"], 6))
print("Recall         :", round(result_model9["recall"], 6))
print("F1-score       :", round(result_model9["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model9,
        valid_pred_model9
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model9["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model9["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model9["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model9["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model9["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'speed_2']

변수 자료형
category      category
amt            float64
trans_hour       int64
age              int64
speed_2        float64
dtype: object

결측치 수
category      0
amt           0
trans_hour    0
age           0
speed_2       0
dtype: int64

speed_2 기초 통계
count    1.296675e+06
mean     8.605355e+01
std      4.590036e+02
min      0.000000e+00
25%      2.968340e+00
50%      1.162299e+01
75%      3.933591e+01
max      1.582018e+04
Name: speed_2, dtype: float64

speed_2 무한대 개수
0

speed_2 상위 분위수
0.900      124.781088
0.950      267.436410
0.990     1403.208303
0.999     7505.267218
1.000    15820.182333
Name: speed_2, dtype: float64

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.0061310581152330445

정상거래 수: 902551
이상거래 수: 5121
scale_pos_weigh

Model 9(speed_2)는 PR-AUC를 조금 개선했지만, 최종 후보로 단독채택할정도는 x

In [17]:
# =========================================================
# category_recent_fraud_rate 1차 누수 점검
# 각 업종의 데이터상 최초 거래 확인
# =========================================================

check_category = (
    df.sort_values("trans_date_trans_time")
    .copy()
)

first_category_transactions = (
    check_category
    .groupby("category", observed=True)
    .head(1)
)

print("업종 수:", len(first_category_transactions))

print("\n업종별 최초 거래의 최근 7일 이상거래율")
print(
    first_category_transactions[
        [
            "category",
            "trans_date_trans_time",
            "is_fraud",
            "category_recent_fraud_rate"
        ]
    ]
    .sort_values("category")
    .to_string(index=False)
)

print("\n최초 거래 변수값 분포")
print(
    first_category_transactions[
        "category_recent_fraud_rate"
    ].value_counts(dropna=False)
)

print("\n결측치 수")
print(
    df["category_recent_fraud_rate"].isna().sum()
)

print("\n기초 통계")
print(
    df["category_recent_fraud_rate"].describe()
)

업종 수: 14

업종별 최초 거래의 최근 7일 이상거래율
      category trans_date_trans_time  is_fraud  category_recent_fraud_rate
 entertainment   2019-01-01 00:00:51         0                         0.0
   food_dining   2019-01-01 00:11:14         0                         0.0
 gas_transport   2019-01-01 00:01:16         0                         0.0
   grocery_net   2019-01-01 00:04:42         0                         0.0
   grocery_pos   2019-01-01 00:00:44         0                         0.0
health_fitness   2019-01-01 12:00:35         0                         0.0
          home   2019-01-01 12:18:39         0                         0.0
     kids_pets   2019-01-01 12:04:54         0                         0.0
      misc_net   2019-01-01 00:00:18         0                         0.0
      misc_pos   2019-01-01 00:03:06         0                         0.0
 personal_care   2019-01-01 12:00:31         0                         0.0
  shopping_net   2019-01-01 00:06:53         0                     

업종별 최초 거래 14건의 is_fraud가 전부 0이기 때문에 데이터 누수 문제가 발생하지 않았다고 할 수 없음. 현재 거래가 잘못 포함됐더라도 최초 거래의 계산값이 0으로 나올 수 있어서, 지금 결과만으로는 구분이 안 됨.

Model 10을 돌리기 전에 category_recent_fraud_rate를 현재 거래를 제외한 과거 7일 데이터로 직접 다시 계산한 값과 비교해야 함. 아래 코드.

In [18]:
# =========================================================
# category_recent_fraud_rate 정밀 누수 점검
# 현재 거래를 제외한 과거 7일 이상거래율 직접 재계산
# =========================================================

check_df = (
    df[
        [
            "category",
            "trans_date_trans_time",
            "is_fraud",
            "category_recent_fraud_rate"
        ]
    ]
    .sort_values("trans_date_trans_time")
    .copy()
)

check_df["trans_date_trans_time"] = pd.to_datetime(
    check_df["trans_date_trans_time"]
)

# 원래 행 위치 보존
check_df["_original_index"] = check_df.index

recalculated_parts = []

for category_name, group in check_df.groupby(
    "category",
    observed=True,
    sort=False
):
    group = (
        group
        .sort_values("trans_date_trans_time")
        .copy()
    )

    group = group.set_index("trans_date_trans_time")

    # 현재 거래를 제외하기 위해 closed="left" 사용
    past_fraud_sum = (
        group["is_fraud"]
        .rolling("7D", closed="left")
        .sum()
    )

    past_transaction_count = (
        group["is_fraud"]
        .rolling("7D", closed="left")
        .count()
    )

    group["recalculated_category_fraud_rate"] = (
        past_fraud_sum
        / past_transaction_count
    ).fillna(0)

    recalculated_parts.append(
        group.reset_index()
    )

recalculated_df = pd.concat(
    recalculated_parts,
    ignore_index=True
)

recalculated_df = (
    recalculated_df
    .set_index("_original_index")
    .sort_index()
)

# 기존 변수와 직접 재계산값의 차이
recalculated_df["rate_difference"] = (
    recalculated_df["category_recent_fraud_rate"]
    - recalculated_df["recalculated_category_fraud_rate"]
).abs()

print("전체 행 수:", len(recalculated_df))

print(
    "\n완전히 일치하는 행 수:",
    np.isclose(
        recalculated_df["category_recent_fraud_rate"],
        recalculated_df["recalculated_category_fraud_rate"],
        atol=1e-12
    ).sum()
)

print(
    "불일치 행 수:",
    (
        ~np.isclose(
            recalculated_df["category_recent_fraud_rate"],
            recalculated_df["recalculated_category_fraud_rate"],
            atol=1e-12
        )
    ).sum()
)

print(
    "\n최대 절대 차이:",
    recalculated_df["rate_difference"].max()
)

print(
    "평균 절대 차이:",
    recalculated_df["rate_difference"].mean()
)

print("\n차이가 큰 행 일부")
print(
    recalculated_df.loc[
        recalculated_df["rate_difference"] > 1e-12,
        [
            "trans_date_trans_time",
            "category",
            "is_fraud",
            "category_recent_fraud_rate",
            "recalculated_category_fraud_rate",
            "rate_difference"
        ]
    ]
    .sort_values(
        "rate_difference",
        ascending=False
    )
    .head(20)
)

전체 행 수: 1296675

완전히 일치하는 행 수: 1296675
불일치 행 수: 0

최대 절대 차이: 9.996344030316351e-17
평균 절대 차이: 4.308784947421991e-17

차이가 큰 행 일부
Empty DataFrame
Columns: [trans_date_trans_time, category, is_fraud, category_recent_fraud_rate, recalculated_category_fraud_rate, rate_difference]
Index: []


ㅇㅋ 데이터 누수 없는 거 확인.

In [19]:
# =========================================================
# Model 10: Baseline + 업종별 최근 7일 이상거래율
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Category Recent Fraud Rate"
FIXED_THRESHOLD = 0.990239

model10_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "category_recent_fraud_rate"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model10_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model10_features))
print("사용 변수:", model10_features)


# 2. 설명변수와 목표변수 생성
X_model10 = df[model10_features].copy()
y_model10 = df["is_fraud"].astype("int8").copy()

X_model10["category"] = (
    X_model10["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model10.dtypes)

print("\n결측치 수")
print(X_model10.isna().sum())

print("\ncategory_recent_fraud_rate 기초 통계")
print(
    X_model10[
        "category_recent_fraud_rate"
    ].describe()
)

print("\n무한대 개수")
print(
    np.isinf(
        X_model10["category_recent_fraud_rate"]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model10 = int(len(df) * 0.70)

X_train_model10 = (
    X_model10
    .iloc[:split_index_model10]
    .copy()
)

X_valid_model10 = (
    X_model10
    .iloc[split_index_model10:]
    .copy()
)

y_train_model10 = (
    y_model10
    .iloc[:split_index_model10]
    .copy()
)

y_valid_model10 = (
    y_model10
    .iloc[split_index_model10:]
    .copy()
)

print("\nTrain:", X_train_model10.shape)
print("Valid:", X_valid_model10.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model10 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model10 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model10:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model10:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model10.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model10.sum())
)

print(
    "Train 이상거래율:",
    y_train_model10.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model10.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model10 = int(
    (y_train_model10 == 0).sum()
)

positive_count_model10 = int(
    (y_train_model10 == 1).sum()
)

scale_pos_weight_model10 = (
    negative_count_model10
    / positive_count_model10
)

print("\n정상거래 수:", negative_count_model10)
print("이상거래 수:", positive_count_model10)
print("scale_pos_weight:", scale_pos_weight_model10)


# 5. 동일 하이퍼파라미터로 모델 생성
model10_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model10
)


# 6. 학습
model10_7030.fit(
    X_train_model10,
    y_train_model10,

    eval_X=X_valid_model10,
    eval_y=y_valid_model10,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model10 = model10_7030.predict_proba(
    X_valid_model10,
    num_iteration=model10_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model10 = (
    valid_prob_model10 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model10 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model10_features),
    "best_iteration": model10_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model10,
        valid_prob_model10
    ),

    "roc_auc": roc_auc_score(
        y_valid_model10,
        valid_prob_model10
    ),

    "precision": precision_score(
        y_valid_model10,
        valid_pred_model10,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model10,
        valid_pred_model10,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model10,
        valid_pred_model10,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 10: "
    "Baseline + Category Recent Fraud Rate 70:30 =========="
)

print("변수 수         :", result_model10["feature_count"])
print("Best iteration :", result_model10["best_iteration"])
print("고정 임계값     :", round(result_model10["threshold"], 6))
print("PR-AUC         :", round(result_model10["pr_auc"], 6))
print("ROC-AUC        :", round(result_model10["roc_auc"], 6))
print("Precision      :", round(result_model10["precision"], 6))
print("Recall         :", round(result_model10["recall"], 6))
print("F1-score       :", round(result_model10["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model10,
        valid_pred_model10
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model10["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model10["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model10["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model10["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model10["f1"]
        - result_7030["f1"],
        6
    )
)

사용 변수 수: 5
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'category_recent_fraud_rate']

변수 자료형
category                      category
amt                            float64
trans_hour                       int64
age                              int64
category_recent_fraud_rate     float64
dtype: object

결측치 수
category                      0
amt                           0
trans_hour                    0
age                           0
category_recent_fraud_rate    0
dtype: int64

category_recent_fraud_rate 기초 통계
count    1.296675e+06
mean     5.763931e-03
std      6.550658e-03
min      0.000000e+00
25%      1.458789e-03
50%      3.311258e-03
75%      7.770472e-03
max      5.326460e-02
Name: category_recent_fraud_rate, dtype: float64

무한대 개수
0

Train: (907672, 5)
Valid: (389003, 5)
Train 기간: 2019-01-01 00:00:18 ~ 2019-12-28 17:18:24
Valid 기간: 2019-12-28 17:18:38 ~ 2020-06-21 12:13:37

Train 이상거래 건수: 5121
Valid 이상거래 건수: 2385
Train 이상거래율: 0.0056419058867079735
Valid 이상거래율: 0.00613105811

Model 10(업종별 최근 7일 내 사기율) 은 제외.

독립 효과 실험 끝. 이제 결과를 기준으로 유효한 변수 조합 모델 시작.

# Model 11. recent_24h_high_amt_count + rolling_sum_amt_1h

In [20]:
# =========================================================
# Model 11:
# Baseline
# + 최근 24시간 고액거래 횟수
# + 최근 1시간 누적 거래금액
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + 24h High Amount Count + 1h Rolling Amount"
FIXED_THRESHOLD = 0.990239

model11_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "rolling_sum_amt_1h"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model11_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model11_features))
print("사용 변수:", model11_features)


# 2. 설명변수와 목표변수 생성
X_model11 = df[model11_features].copy()
y_model11 = df["is_fraud"].astype("int8").copy()

X_model11["category"] = (
    X_model11["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model11.dtypes)

print("\n결측치 수")
print(X_model11.isna().sum())

print("\n추가 변수 기초 통계")
print(
    X_model11[
        [
            "recent_24h_high_amt_count",
            "rolling_sum_amt_1h"
        ]
    ].describe()
)

print("\n무한대 개수")
print(
    np.isinf(
        X_model11[
            [
                "recent_24h_high_amt_count",
                "rolling_sum_amt_1h"
            ]
        ]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model11 = int(len(df) * 0.70)

X_train_model11 = (
    X_model11
    .iloc[:split_index_model11]
    .copy()
)

X_valid_model11 = (
    X_model11
    .iloc[split_index_model11:]
    .copy()
)

y_train_model11 = (
    y_model11
    .iloc[:split_index_model11]
    .copy()
)

y_valid_model11 = (
    y_model11
    .iloc[split_index_model11:]
    .copy()
)

print("\nTrain:", X_train_model11.shape)
print("Valid:", X_valid_model11.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model11 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model11 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model11:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model11:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model11.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model11.sum())
)

print(
    "Train 이상거래율:",
    y_train_model11.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model11.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model11 = int(
    (y_train_model11 == 0).sum()
)

positive_count_model11 = int(
    (y_train_model11 == 1).sum()
)

scale_pos_weight_model11 = (
    negative_count_model11
    / positive_count_model11
)

print("\n정상거래 수:", negative_count_model11)
print("이상거래 수:", positive_count_model11)
print("scale_pos_weight:", scale_pos_weight_model11)


# 5. 동일 하이퍼파라미터로 모델 생성
model11_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model11
)


# 6. 학습
model11_7030.fit(
    X_train_model11,
    y_train_model11,

    eval_X=X_valid_model11,
    eval_y=y_valid_model11,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model11 = model11_7030.predict_proba(
    X_valid_model11,
    num_iteration=model11_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model11 = (
    valid_prob_model11 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model11 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model11_features),
    "best_iteration": model11_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model11,
        valid_prob_model11
    ),

    "roc_auc": roc_auc_score(
        y_valid_model11,
        valid_prob_model11
    ),

    "precision": precision_score(
        y_valid_model11,
        valid_pred_model11,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model11,
        valid_pred_model11,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model11,
        valid_pred_model11,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 11: "
    "Baseline + 24h High Amount Count "
    "+ 1h Rolling Amount 70:30 =========="
)

print("변수 수         :", result_model11["feature_count"])
print("Best iteration :", result_model11["best_iteration"])
print("고정 임계값     :", round(result_model11["threshold"], 6))
print("PR-AUC         :", round(result_model11["pr_auc"], 6))
print("ROC-AUC        :", round(result_model11["roc_auc"], 6))
print("Precision      :", round(result_model11["precision"], 6))
print("Recall         :", round(result_model11["recall"], 6))
print("F1-score       :", round(result_model11["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model11,
        valid_pred_model11
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model11["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model11["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model11["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model11["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model11["f1"]
        - result_7030["f1"],
        6
    )
)


# 12. Model 8 대비 변화
print(
    "\n========== Model 8 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model11["pr_auc"]
        - result_model8["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model11["roc_auc"]
        - result_model8["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model11["precision"]
        - result_model8["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model11["recall"]
        - result_model8["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model11["f1"]
        - result_model8["f1"],
        6
    )
)

사용 변수 수: 6
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'rolling_sum_amt_1h']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
recent_24h_high_amt_count       int64
rolling_sum_amt_1h            float64
dtype: object

결측치 수
category                     0
amt                          0
trans_hour                   0
age                          0
recent_24h_high_amt_count    0
rolling_sum_amt_1h           0
dtype: int64

추가 변수 기초 통계
       recent_24h_high_amt_count  rolling_sum_amt_1h
count               1.296675e+06        1.296675e+06
mean                4.816936e-02        8.491117e+01
std                 2.741523e-01        1.947081e+02
min                 0.000000e+00        1.000000e+00
25%                 0.000000e+00        1.425000e+01
50%                 0.000000e+00        5.429000e+01
75%                 0.000000e+00        9.630000e+

Model 8보다는 F1이 미세하게 성능 떨어짐. PR-AUC와 Precision은 조금 좋아짐. 

# Model 12. baseline + recent_24h_high_amt_count + amt_to_prior_median_ratio

가장 강력했던 단독 변수를 조합

In [22]:
# =========================================================
# Model 12:
# Baseline
# + 최근 24시간 고액거래 횟수
# + 개인별 평소 금액 대비 거래금액 비율
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + 24h High Amount Count + Personalized Amount Ratio"
FIXED_THRESHOLD = 0.990239

model12_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model12_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model12_features))
print("사용 변수:", model12_features)


# 2. 설명변수와 목표변수 생성
X_model12 = df[model12_features].copy()
y_model12 = df["is_fraud"].astype("int8").copy()

X_model12["category"] = (
    X_model12["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model12.dtypes)

print("\n결측치 수")
print(X_model12.isna().sum())

print("\n추가 변수 기초 통계")
print(
    X_model12[
        [
            "recent_24h_high_amt_count",
            "amt_to_prior_median_ratio"
        ]
    ].describe()
)

print("\n무한대 개수")
print(
    np.isinf(
        X_model12[
            [
                "recent_24h_high_amt_count",
                "amt_to_prior_median_ratio"
            ]
        ]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model12 = int(len(df) * 0.70)

X_train_model12 = (
    X_model12
    .iloc[:split_index_model12]
    .copy()
)

X_valid_model12 = (
    X_model12
    .iloc[split_index_model12:]
    .copy()
)

y_train_model12 = (
    y_model12
    .iloc[:split_index_model12]
    .copy()
)

y_valid_model12 = (
    y_model12
    .iloc[split_index_model12:]
    .copy()
)

print("\nTrain:", X_train_model12.shape)
print("Valid:", X_valid_model12.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model12 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model12 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model12:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model12:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model12.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model12.sum())
)

print(
    "Train 이상거래율:",
    y_train_model12.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model12.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model12 = int(
    (y_train_model12 == 0).sum()
)

positive_count_model12 = int(
    (y_train_model12 == 1).sum()
)

scale_pos_weight_model12 = (
    negative_count_model12
    / positive_count_model12
)

print("\n정상거래 수:", negative_count_model12)
print("이상거래 수:", positive_count_model12)
print("scale_pos_weight:", scale_pos_weight_model12)


# 5. 동일 하이퍼파라미터로 모델 생성
model12_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model12
)


# 6. 학습
model12_7030.fit(
    X_train_model12,
    y_train_model12,

    eval_X=X_valid_model12,
    eval_y=y_valid_model12,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model12 = model12_7030.predict_proba(
    X_valid_model12,
    num_iteration=model12_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model12 = (
    valid_prob_model12 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model12 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model12_features),
    "best_iteration": model12_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model12,
        valid_prob_model12
    ),

    "roc_auc": roc_auc_score(
        y_valid_model12,
        valid_prob_model12
    ),

    "precision": precision_score(
        y_valid_model12,
        valid_pred_model12,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model12,
        valid_pred_model12,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model12,
        valid_pred_model12,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 12: "
    "Baseline + 24h High Amount Count "
    "+ Personalized Amount Ratio 70:30 =========="
)

print("변수 수         :", result_model12["feature_count"])
print("Best iteration :", result_model12["best_iteration"])
print("고정 임계값     :", round(result_model12["threshold"], 6))
print("PR-AUC         :", round(result_model12["pr_auc"], 6))
print("ROC-AUC        :", round(result_model12["roc_auc"], 6))
print("Precision      :", round(result_model12["precision"], 6))
print("Recall         :", round(result_model12["recall"], 6))
print("F1-score       :", round(result_model12["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model12,
        valid_pred_model12
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model12["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model12["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model12["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model12["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model12["f1"]
        - result_7030["f1"],
        6
    )
)


# 12. Model 8 대비 변화
print(
    "\n========== Model 8 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model12["pr_auc"]
        - result_model8["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model12["roc_auc"]
        - result_model8["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model12["precision"]
        - result_model8["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model12["recall"]
        - result_model8["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model12["f1"]
        - result_model8["f1"],
        6
    )
)

사용 변수 수: 6
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'amt_to_prior_median_ratio']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
recent_24h_high_amt_count       int64
amt_to_prior_median_ratio     float64
dtype: object

결측치 수
category                        0
amt                             0
trans_hour                      0
age                             0
recent_24h_high_amt_count       0
amt_to_prior_median_ratio    1649
dtype: int64

추가 변수 기초 통계
       recent_24h_high_amt_count  amt_to_prior_median_ratio
count               1.296675e+06               1.295026e+06
mean                4.816936e-02               1.643860e+00
std                 2.741523e-01               4.558937e+00
min                 0.000000e+00               2.176430e-03
25%                 0.000000e+00               2.606922e-01
50%                 0.000000e+00    

Model 12가 현재 1위.

# Model 13. baseline + amt_to_prior_median_ratio + rolling_sum_amt_1h

In [23]:
# =========================================================
# Model 13:
# Baseline
# + 개인별 평소 금액 대비 거래금액 비율
# + 최근 1시간 누적 거래금액
# 시간순 70:30
# =========================================================

MODEL_NAME = "Baseline + Personalized Amount Ratio + 1h Rolling Amount"
FIXED_THRESHOLD = 0.990239

model13_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model13_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model13_features))
print("사용 변수:", model13_features)


# 2. 설명변수와 목표변수 생성
X_model13 = df[model13_features].copy()
y_model13 = df["is_fraud"].astype("int8").copy()

X_model13["category"] = (
    X_model13["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model13.dtypes)

print("\n결측치 수")
print(X_model13.isna().sum())

print("\n추가 변수 기초 통계")
print(
    X_model13[
        [
            "amt_to_prior_median_ratio",
            "rolling_sum_amt_1h"
        ]
    ].describe()
)

print("\n무한대 개수")
print(
    np.isinf(
        X_model13[
            [
                "amt_to_prior_median_ratio",
                "rolling_sum_amt_1h"
            ]
        ]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model13 = int(len(df) * 0.70)

X_train_model13 = (
    X_model13
    .iloc[:split_index_model13]
    .copy()
)

X_valid_model13 = (
    X_model13
    .iloc[split_index_model13:]
    .copy()
)

y_train_model13 = (
    y_model13
    .iloc[:split_index_model13]
    .copy()
)

y_valid_model13 = (
    y_model13
    .iloc[split_index_model13:]
    .copy()
)

print("\nTrain:", X_train_model13.shape)
print("Valid:", X_valid_model13.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model13 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model13 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model13:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model13:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model13.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model13.sum())
)

print(
    "Train 이상거래율:",
    y_train_model13.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model13.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model13 = int(
    (y_train_model13 == 0).sum()
)

positive_count_model13 = int(
    (y_train_model13 == 1).sum()
)

scale_pos_weight_model13 = (
    negative_count_model13
    / positive_count_model13
)

print("\n정상거래 수:", negative_count_model13)
print("이상거래 수:", positive_count_model13)
print("scale_pos_weight:", scale_pos_weight_model13)


# 5. 동일 하이퍼파라미터로 모델 생성
model13_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model13
)


# 6. 학습
model13_7030.fit(
    X_train_model13,
    y_train_model13,

    eval_X=X_valid_model13,
    eval_y=y_valid_model13,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model13 = model13_7030.predict_proba(
    X_valid_model13,
    num_iteration=model13_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model13 = (
    valid_prob_model13 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model13 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model13_features),
    "best_iteration": model13_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model13,
        valid_prob_model13
    ),

    "roc_auc": roc_auc_score(
        y_valid_model13,
        valid_prob_model13
    ),

    "precision": precision_score(
        y_valid_model13,
        valid_pred_model13,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model13,
        valid_pred_model13,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model13,
        valid_pred_model13,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 13: "
    "Baseline + Personalized Amount Ratio "
    "+ 1h Rolling Amount 70:30 =========="
)

print("변수 수         :", result_model13["feature_count"])
print("Best iteration :", result_model13["best_iteration"])
print("고정 임계값     :", round(result_model13["threshold"], 6))
print("PR-AUC         :", round(result_model13["pr_auc"], 6))
print("ROC-AUC        :", round(result_model13["roc_auc"], 6))
print("Precision      :", round(result_model13["precision"], 6))
print("Recall         :", round(result_model13["recall"], 6))
print("F1-score       :", round(result_model13["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model13,
        valid_pred_model13
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model13["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model13["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model13["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model13["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model13["f1"]
        - result_7030["f1"],
        6
    )
)


# 12. Model 4 대비 변화
print(
    "\n========== Model 4 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model13["pr_auc"]
        - result_model4["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model13["roc_auc"]
        - result_model4["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model13["precision"]
        - result_model4["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model13["recall"]
        - result_model4["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model13["f1"]
        - result_model4["f1"],
        6
    )
)


# 13. Model 7 대비 변화
print(
    "\n========== Model 7 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model13["pr_auc"]
        - result_model7["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model13["roc_auc"]
        - result_model7["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model13["precision"]
        - result_model7["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model13["recall"]
        - result_model7["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model13["f1"]
        - result_model7["f1"],
        6
    )
)

사용 변수 수: 6
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'amt_to_prior_median_ratio', 'rolling_sum_amt_1h']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
amt_to_prior_median_ratio     float64
rolling_sum_amt_1h            float64
dtype: object

결측치 수
category                        0
amt                             0
trans_hour                      0
age                             0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
dtype: int64

추가 변수 기초 통계
       amt_to_prior_median_ratio  rolling_sum_amt_1h
count               1.295026e+06        1.296675e+06
mean                1.643860e+00        8.491117e+01
std                 4.558937e+00        1.947081e+02
min                 2.176430e-03        1.000000e+00
25%                 2.606922e-01        1.425000e+01
50%                 1.003543e+00        5.429000e+01
75%                 1.894498e+00

Model 13 효과 있음. 단독으로 썼을 때보다 모든 핵심 지표가 좋아짐.

# Model 14. 

- recent_24h_high_amt_count,
- amt_to_prior_median_ratio
- rolling_sum_amt_1h

In [ ]:
# =========================================================
# Model 14:
# Baseline
# + 최근 24시간 고액거래 횟수
# + 개인별 평소 금액 대비 거래금액 비율
# + 최근 1시간 누적 거래금액
# 시간순 70:30
# =========================================================

MODEL_NAME = (
    "Baseline + 24h High Amount Count "
    "+ Personalized Amount Ratio "
    "+ 1h Rolling Amount"
)

FIXED_THRESHOLD = 0.990239

model14_features = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h"
]


# 1. 변수 존재 여부 확인
missing_columns = [
    col for col in model14_features
    if col not in df.columns
]

if missing_columns:
    raise KeyError(f"데이터에 없는 변수: {missing_columns}")

print("사용 변수 수:", len(model14_features))
print("사용 변수:", model14_features)


# 2. 설명변수와 목표변수 생성
X_model14 = df[model14_features].copy()
y_model14 = df["is_fraud"].astype("int8").copy()

X_model14["category"] = (
    X_model14["category"]
    .astype("category")
)

print("\n변수 자료형")
print(X_model14.dtypes)

print("\n결측치 수")
print(X_model14.isna().sum())

print("\n추가 변수 기초 통계")
print(
    X_model14[
        [
            "recent_24h_high_amt_count",
            "amt_to_prior_median_ratio",
            "rolling_sum_amt_1h"
        ]
    ].describe()
)

print("\n무한대 개수")
print(
    np.isinf(
        X_model14[
            [
                "recent_24h_high_amt_count",
                "amt_to_prior_median_ratio",
                "rolling_sum_amt_1h"
            ]
        ]
    ).sum()
)


# 3. 시간순 70:30 분할
split_index_model14 = int(len(df) * 0.70)

X_train_model14 = (
    X_model14
    .iloc[:split_index_model14]
    .copy()
)

X_valid_model14 = (
    X_model14
    .iloc[split_index_model14:]
    .copy()
)

y_train_model14 = (
    y_model14
    .iloc[:split_index_model14]
    .copy()
)

y_valid_model14 = (
    y_model14
    .iloc[split_index_model14:]
    .copy()
)

print("\nTrain:", X_train_model14.shape)
print("Valid:", X_valid_model14.shape)

print(
    "Train 기간:",
    df.loc[
        :split_index_model14 - 1,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        :split_index_model14 - 1,
        "trans_date_trans_time"
    ].max()
)

print(
    "Valid 기간:",
    df.loc[
        split_index_model14:,
        "trans_date_trans_time"
    ].min(),
    "~",
    df.loc[
        split_index_model14:,
        "trans_date_trans_time"
    ].max()
)

print(
    "\nTrain 이상거래 건수:",
    int(y_train_model14.sum())
)

print(
    "Valid 이상거래 건수:",
    int(y_valid_model14.sum())
)

print(
    "Train 이상거래율:",
    y_train_model14.mean()
)

print(
    "Valid 이상거래율:",
    y_valid_model14.mean()
)


# 4. 클래스 불균형 가중치 계산
negative_count_model14 = int(
    (y_train_model14 == 0).sum()
)

positive_count_model14 = int(
    (y_train_model14 == 1).sum()
)

scale_pos_weight_model14 = (
    negative_count_model14
    / positive_count_model14
)

print("\n정상거래 수:", negative_count_model14)
print("이상거래 수:", positive_count_model14)
print("scale_pos_weight:", scale_pos_weight_model14)


# 5. 동일 하이퍼파라미터로 모델 생성
model14_7030 = lgb.LGBMClassifier(
    **MODEL_PARAMS,
    scale_pos_weight=scale_pos_weight_model14
)


# 6. 학습
model14_7030.fit(
    X_train_model14,
    y_train_model14,

    eval_X=X_valid_model14,
    eval_y=y_valid_model14,

    eval_metric="average_precision",

    categorical_feature=["category"],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            first_metric_only=True,
            verbose=True
        ),
        lgb.log_evaluation(period=50)
    ]
)


# 7. 검증셋 예측확률
valid_prob_model14 = model14_7030.predict_proba(
    X_valid_model14,
    num_iteration=model14_7030.best_iteration_
)[:, 1]


# 8. Baseline 고정 임계값 적용
valid_pred_model14 = (
    valid_prob_model14 >= FIXED_THRESHOLD
).astype("int8")


# 9. 성능 평가
result_model14 = {
    "model": MODEL_NAME,
    "split": "70:30",
    "feature_count": len(model14_features),
    "best_iteration": model14_7030.best_iteration_,
    "threshold": FIXED_THRESHOLD,

    "pr_auc": average_precision_score(
        y_valid_model14,
        valid_prob_model14
    ),

    "roc_auc": roc_auc_score(
        y_valid_model14,
        valid_prob_model14
    ),

    "precision": precision_score(
        y_valid_model14,
        valid_pred_model14,
        zero_division=0
    ),

    "recall": recall_score(
        y_valid_model14,
        valid_pred_model14,
        zero_division=0
    ),

    "f1": f1_score(
        y_valid_model14,
        valid_pred_model14,
        zero_division=0
    )
}


# 10. 결과 출력
print(
    "\n========== Model 14: "
    "Baseline + 24h High Amount Count "
    "+ Personalized Amount Ratio "
    "+ 1h Rolling Amount 70:30 =========="
)

print("변수 수         :", result_model14["feature_count"])
print("Best iteration :", result_model14["best_iteration"])
print("고정 임계값     :", round(result_model14["threshold"], 6))
print("PR-AUC         :", round(result_model14["pr_auc"], 6))
print("ROC-AUC        :", round(result_model14["roc_auc"], 6))
print("Precision      :", round(result_model14["precision"], 6))
print("Recall         :", round(result_model14["recall"], 6))
print("F1-score       :", round(result_model14["f1"], 6))

print("\n혼동행렬")
print(
    confusion_matrix(
        y_valid_model14,
        valid_pred_model14
    )
)


# 11. Baseline 대비 변화
print("\n========== Baseline 대비 변화 ==========")

print(
    "PR-AUC 변화:",
    round(
        result_model14["pr_auc"]
        - result_7030["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model14["roc_auc"]
        - result_7030["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model14["precision"]
        - result_7030["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model14["recall"]
        - result_7030["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model14["f1"]
        - result_7030["f1"],
        6
    )
)


# 12. 현재 1위 Model 12 대비 변화
print(
    "\n========== Model 12 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model14["pr_auc"]
        - result_model12["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model14["roc_auc"]
        - result_model12["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model14["precision"]
        - result_model12["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model14["recall"]
        - result_model12["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model14["f1"]
        - result_model12["f1"],
        6
    )
)


# 13. Model 11 대비 변화
print(
    "\n========== Model 11 대비 변화 =========="
)

print(
    "PR-AUC 변화:",
    round(
        result_model14["pr_auc"]
        - result_model11["pr_auc"],
        6
    )
)

print(
    "ROC-AUC 변화:",
    round(
        result_model14["roc_auc"]
        - result_model11["roc_auc"],
        6
    )
)

print(
    "Precision 변화:",
    round(
        result_model14["precision"]
        - result_model11["precision"],
        6
    )
)

print(
    "Recall 변화:",
    round(
        result_model14["recall"]
        - result_model11["recall"],
        6
    )
)

print(
    "F1 변화:",
    round(
        result_model14["f1"]
        - result_model11["f1"],
        6
    )
)

사용 변수 수: 7
사용 변수: ['category', 'amt', 'trans_hour', 'age', 'recent_24h_high_amt_count', 'amt_to_prior_median_ratio', 'rolling_sum_amt_1h']

변수 자료형
category                     category
amt                           float64
trans_hour                      int64
age                             int64
recent_24h_high_amt_count       int64
amt_to_prior_median_ratio     float64
rolling_sum_amt_1h            float64
dtype: object

결측치 수
category                        0
amt                             0
trans_hour                      0
age                             0
recent_24h_high_amt_count       0
amt_to_prior_median_ratio    1649
rolling_sum_amt_1h              0
dtype: int64

추가 변수 기초 통계
       recent_24h_high_amt_count  amt_to_prior_median_ratio  \
count               1.296675e+06               1.295026e+06   
mean                4.816936e-02               1.643860e+00   
std                 2.741523e-01               4.558937e+00   
min                 0.000000e+00               2.1

: 